In [ ]:
!pip install geopandas geopy

In [5]:
import pandas as pd
import unicodedata
import re
import geopandas as gpd
from geopy.distance import geodesic

ModuleNotFoundError: No module named 'geopandas'

In [ ]:
gdf_bairros = gpd.read_file("bairros.geojson")
gdf_bairros

,OBJECTID,CBAIRRCODI,VBAIRROID,EBAIRRNOME,CRPAAACODI,CMICROCODI,TBAIRRULAT,CEMPRECODI,AUSUACMATR,EBAIRRNOMEOF,EBAIRRLINK,TBAIRRSULAT,DB2GSE.ST_Area(SHAPE),DB2GSE.SdeLength(SHAPE),geometry
0,1,752,None,CURADO,5,3,2019-10-15 21:00:00,None,None,Curado,None,None,8.255774e+06,18101.174156,"MULTIPOLYGON (((-34.9712 -8.08154, -34.9712 -8..."
1,2,396,None,DOIS UNIDOS,2,3,2019-10-15 21:00:00,None,None,Dois Unidos,None,None,3.118840e+06,8101.114056,"MULTIPOLYGON (((-34.91022 -7.99242, -34.9102 -..."
2,3,132,None,AFLITOS,3,1,2019-10-15 21:00:00,None,None,Aflitos,None,None,3.074612e+05,2417.227267,"MULTIPOLYGON (((-34.89256 -8.0399, -34.89248 -..."
3,4,906,None,SANCHO,5,3,2019-10-15 21:00:00,None,None,Sancho,None,None,6.324592e+05,3188.997249,"MULTIPOLYGON (((-34.96576 -8.08146, -34.96576 ..."
4,5,736,None,VARZEA,4,3,2019-10-15 21:00:00,None,None,Várzea,None,None,2.239116e+07,27311.242636,"MULTIPOLYGON (((-34.96607 -8.02122, -34.96604 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,91,779,None,AFOGADOS,5,1,2019-10-15 21:00:00,None,None,Afogados,None,None,3.385466e+06,9819.407396,"MULTIPOLYGON (((-34.90918 -8.06518, -34.90898 ..."
90,92,507,None,APIPUCOS,3,1,2019-10-15 21:00:00,None,None,Apipucos,None,None,1.265252e+06,7287.550605,"MULTIPOLYGON (((-34.93403 -8.01838, -34.93389 ..."
91,93,620,None,CORREGO DO JENIPAPO,3,3,2019-10-15 21:00:00,None,None,Córrego do Jenipapo,None,None,5.702820e+05,4232.373337,"MULTIPOLYGON (((-34.93739 -7.99509, -34.93733 ..."
92,94,930,None,PONTO DE PARADA,2,1,2019-10-15 21:00:00,None,None,Ponto de Parada,None,None,1.852260e+05,1876.069588,"MULTIPOLYGON (((-34.8915 -8.03113, -34.89227 -..."


In [ ]:
gdf = gdf_bairros.copy()
gdf = gdf.to_crs(epsg=4326)
gdf["nome"] = gdf["EBAIRRNOMEOF"]
gdf["centroid"] = gdf.geometry.centroid
centroid_dict = gdf.set_index("nome")["centroid"].apply(lambda p: (p.y, p.x)).to_dict()
dados_vizinhos = []
for idx, row in gdf.iterrows():
    nome = row["nome"]
    geom = row.geometry
    vizinhos = gdf[gdf.geometry.touches(geom)]
    for _, vizinho_row in vizinhos.iterrows():
        vizinho_nome = vizinho_row["nome"]
        coord_a = centroid_dict[nome]
        coord_b = centroid_dict[vizinho_nome]
        distancia = geodesic(coord_a, coord_b).meters
        dados_vizinhos.append({
            "bairro_origem": nome,
            "bairro_destino": vizinho_nome,
            "peso": round(distancia, 2)
        })
df_vizinhos = pd.DataFrame(dados_vizinhos)


/tmp/ipython-input-592542132.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid"] = gdf.geometry.centroid


In [ ]:
df_vizinhos['par_bairros_ordenado'] = df_vizinhos.apply(
    lambda row: tuple(sorted((row['bairro_origem'], row['bairro_destino']))),
    axis=1
)

df_vizinhos_unique = df_vizinhos.drop_duplicates(subset=['par_bairros_ordenado']).drop(columns=['par_bairros_ordenado'])



(242, 3)

In [6]:

def padronizar_texto(texto):
    # Remove acentos
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    # Remove caracteres especiais e mantém apenas letras, números e espaços
    texto = re.sub(r'[^a-zA-Z0-9\s]', '', texto)
    # Substitui múltiplos espaços por um único espaço e remove espaços no início/fim
    texto = re.sub(r'\s+', ' ', texto).strip()
    texto = texto.lower()
    return texto

df_vizinhos_unique['bairro_origem'] = df_vizinhos_unique['bairro_origem'].apply(padronizar_texto)
df_vizinhos_unique['bairro_destino'] = df_vizinhos_unique['bairro_destino'].apply(padronizar_texto)

NameError: name 'df_vizinhos_unique' is not defined

In [ ]:
df_logradouro = pd.read_csv("trechologradouro.csv", sep=';')
df_logradouro.shape

(11921, 10)

In [ ]:
codi_bairro_counts = df_logradouro.groupby('codlogradouro')['nomeBairro'].nunique()

codis_with_different_bairros = codi_bairro_counts[codi_bairro_counts > 1].index

df_logradouro_different_bairros = df_logradouro[df_logradouro['codlogradouro'].isin(codis_with_different_bairros)]

display(df_logradouro_different_bairros.head())
display(df_logradouro_different_bairros.shape)

,codlogradouro,nome_logradouro_concatenado,nome_oficial_logradouro,nome_logradouro_resumido,cod_indica_pavimentacao,desc_indica_pavimentacao,indica_corredor_transporte,indica_perimetral,codbairro,nomeBairro
12,21148,RUA ELIZEU CESAR,Rua Elizeu César,R. Elizeu Cesar,X,Não definida,NaN,NaN,850,AREIAS
13,21148,RUA ELIZEU CESAR,Rua Elizeu César,R. Elizeu Cesar,X,Não definida,NaN,NaN,825,JIQUIA
19,28185,RUA GUEDES PEREIRA,Rua Guedes Pereira,R. Guedes Pereira,S,Via Pavimentada,NaN,NaN,191,TAMARINEIRA
20,28185,RUA GUEDES PEREIRA,Rua Guedes Pereira,R. Guedes Pereira,S,Via Pavimentada,NaN,NaN,434,CASA AMARELA
36,64408,RUA CAPITAO VICENTE DA MOTA,Rua Capitão Vicente da Mota,R. Cap Vicente da Mota,S,Via Pavimentada,NaN,NaN,205,BOA VIAGEM


(3189, 10)

In [ ]:
def aggregate_bairros(group):
    bairros = group['nomeBairro'].unique()
    if len(bairros) >= 2:
        group['nomeBairro_duplicata'] = bairros[1]
        return group.iloc[[0]]
    else:
        group['nomeBairro_duplicata'] = None
        return group.iloc[[0]]

df_logradouro_unified = df_logradouro_different_bairros.groupby('codlogradouro').apply(aggregate_bairros).reset_index(drop=True)

df_logradouro_filtered = df_logradouro_unified.dropna(subset=['nomeBairro_duplicata']).copy()

df_logradouro_filtered['par_bairros_ordenado'] = df_logradouro_filtered.apply(
    lambda row: tuple(sorted((row['nomeBairro'], row['nomeBairro_duplicata']))),
    axis=1
)

df_logradouro_final = df_logradouro_filtered.drop_duplicates(subset=['par_bairros_ordenado']).drop(columns=['par_bairros_ordenado'])

df_logradouro_final['nomeBairro'] = df_logradouro_final['nomeBairro'].apply(padronizar_texto)
df_logradouro_final['nomeBairro_duplicata'] = df_logradouro_final['nomeBairro_duplicata'].apply(padronizar_texto)


csv_path = "logradouro_final.csv"
df_logradouro_final.to_csv(csv_path, index=False)

/tmp/ipython-input-3834003039.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_logradouro_unified = df_logradouro_different_bairros.groupby('codlogradouro').apply(aggregate_bairros).reset_index(drop=True)


In [22]:
df_vizinhos_unique_ordered1 = df_vizinhos_unique.copy()
df_vizinhos_unique_ordered2 = df_vizinhos_unique.copy()

df_vizinhos_unique_ordered1 = df_vizinhos_unique_ordered1.rename(
    columns={'bairro_origem': 'bairro1', 'bairro_destino': 'bairro2'}
)

df_vizinhos_unique_ordered2 = df_vizinhos_unique_ordered2.rename(
    columns={'bairro_origem': 'bairro2', 'bairro_destino': 'bairro1'}
)


df_logradouro_final_renamed = df_logradouro_final.rename(
    columns={'nomeBairro': 'bairro1', 'nomeBairro_duplicata': 'bairro2'}
)


df_vizinhos_merged1 = pd.merge(
    df_vizinhos_unique_ordered1,
    df_logradouro_final_renamed[['bairro1', 'bairro2', 'nome_logradouro_resumido']],
    on=['bairro1', 'bairro2'],
    how='left'
)


df_vizinhos_merged2 = pd.merge(
    df_vizinhos_unique_ordered2,
    df_logradouro_final_renamed[['bairro1', 'bairro2', 'nome_logradouro_resumido']],
    on=['bairro1', 'bairro2'],
    how='left'
)


df_vizinhos_merged = df_vizinhos_merged1.copy()
df_vizinhos_merged['nome_logradouro_resumido'] = df_vizinhos_merged1['nome_logradouro_resumido'].fillna(df_vizinhos_merged2['nome_logradouro_resumido'])

df_vizinhos_merged = df_vizinhos_merged.rename(columns={'nome_logradouro_resumido': 'logradouro',
                                                        'bairro1': 'bairro_origem',
                                                        'bairro2': 'bairro_destino'})

df_vizinhos_merged = df_vizinhos_merged.drop(columns=['par_bairros_ordenado'])
df_vizinhos_merged = df_vizinhos_merged.dropna(subset=['logradouro'])

display(df_vizinhos_merged)
display(df_vizinhos_merged.shape)

contagem_nulos = df_vizinhos_merged['logradouro'].isna().sum()
display(contagem_nulos)

csv_path = "adjacencias_bairros.csv"
df_vizinhos_merged.to_csv(csv_path, index=False)

,bairro_origem,bairro_destino,peso,logradouro
0,curado,sancho,1499.40,Av Liberdade
1,curado,varzea,3178.48,R. Min Mario Andreazza
2,curado,toto,1578.28,R. Ana Albuquerque Lima
3,curado,engenho do meio,2469.76,R. José Dos Santos
5,curado,torroes,2715.07,R. Brejo Novo
...,...,...,...,...
234,brejo de beberibe,nova descoberta,872.98,R. Agrolandia
235,nova descoberta,corrego do jenipapo,988.43,R. Alvares Florense
236,monteiro,apipucos,1224.80,R. de Apipucos
239,imbiribeira,afogados,3055.85,R. Alvorada do Norte


(197, 4)

np.int64(0)

In [7]:
df_correcao = pd.read_csv("bairros_unique.csv")
df_correcao['bairro'] = df_correcao['bairro'].apply(padronizar_texto)

df_correcao.to_csv("bairros_unique.csv", index=False) 